# Data Understanding

## Objective

Understand the structure, size, columns, data types, and initial quality of the raw salary dataset before cleaning or analysis.

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
current_path = Path.cwd()

current_path

WindowsPath('c:/Users/aydna/global-ai-ml-salary-analysis/notebooks')

In [3]:
project_root = current_path

if project_root.name == "notebooks":
    project_root = project_root.parent

project_root    

WindowsPath('c:/Users/aydna/global-ai-ml-salary-analysis')

In [4]:
raw_file_path = project_root /"data" / "raw" / "salaries.csv"

raw_file_path
raw_file_path.exists()

True

In [5]:
df_raw = pd.read_csv(raw_file_path)

df_raw.shape
df_raw.head()
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151445 entries, 0 to 151444
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   work_year           151445 non-null  int64 
 1   experience_level    151445 non-null  object
 2   employment_type     151445 non-null  object
 3   job_title           151445 non-null  object
 4   salary              151445 non-null  int64 
 5   salary_currency     151445 non-null  object
 6   salary_in_usd       151445 non-null  int64 
 7   employee_residence  151445 non-null  object
 8   remote_ratio        151445 non-null  int64 
 9   company_location    151445 non-null  object
 10  company_size        151445 non-null  object
dtypes: int64(4), object(7)
memory usage: 12.7+ MB


## Initial Data Quality Assessment

In [6]:
missing_values = df_raw.isnull().sum()

missing_values

work_year             0
experience_level      0
employment_type       0
job_title             0
salary                0
salary_currency       0
salary_in_usd         0
employee_residence    0
remote_ratio          0
company_location      0
company_size          0
dtype: int64

In [7]:
duplicate_rows = df_raw.duplicated().sum()

duplicate_rows

np.int64(79532)

In [8]:
unique_values = df_raw.nunique().sort_values()

unique_values

remote_ratio              3
company_size              3
experience_level          4
employment_type           4
work_year                 6
salary_currency          26
company_location         97
employee_residence      104
job_title               422
salary                12308
salary_in_usd         13580
dtype: int64

In [9]:
year_counts = df_raw["work_year"].value_counts().sort_index()

year_counts

work_year
2020       75
2021      218
2022     1661
2023     8524
2024    62241
2025    78726
Name: count, dtype: int64

In [10]:
df_raw["experience_level"].value_counts()

experience_level
SE    87491
MI    46128
EN    13663
EX     4163
Name: count, dtype: int64

In [11]:
duplicate_percentage = duplicate_rows / len(df_raw) * 100

round(duplicate_percentage, 2)

np.float64(52.52)

In [12]:
duplicate_mask = df_raw.duplicated(keep=False)

duplicate_mask.head()

0    False
1    False
2    False
3    False
4     True
dtype: bool

In [13]:
duplicate_examples = df_raw[duplicate_mask]

duplicate_examples.head(5)

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
4,2025,MI,FT,Engineer,160000,USD,160000,US,100,US,M
5,2025,MI,FT,Engineer,140000,USD,140000,US,100,US,M
6,2025,SE,FT,AI Product Lead,175000,USD,175000,US,100,US,M
7,2025,SE,FT,AI Product Lead,152900,USD,152900,US,100,US,M
8,2025,SE,FT,AI Engineer,97900,USD,97900,US,100,US,M


In [14]:
categorical_columns = [
     "employment_type",
    "remote_ratio",
    "company_size"
]

for column in categorical_columns:
    print(f"\n --- {column} ---")
    print(df_raw[column].value_counts())


 --- employment_type ---
employment_type
FT    150541
CT       467
PT       421
FL        16
Name: count, dtype: int64

 --- remote_ratio ---
remote_ratio
0      119570
100     31546
50        329
Name: count, dtype: int64

 --- company_size ---
company_size
M    147302
L      3926
S       217
Name: count, dtype: int64


## Salary Data Profiling

In [15]:
salary_summary = (
    df_raw[["salary", "salary_in_usd"]]
    .describe()
    .round(2)

)

salary_summary

,salary,salary_in_usd
count,151445.00,151445.00
mean,162837.96,157527.46
std,208012.40,74150.77
min,14000.00,15000.00
25%,106000.00,105800.00
50%,147000.00,146100.00
75%,199000.00,198000.00
max,30400000.00,800000.00


In [16]:
invalid_salary_mask = df_raw["salary_in_usd"] <= 0 

invalid_salary_count = invalid_salary_mask.sum()

invalid_salary_count

np.int64(0)

In [17]:
lowest_salaries = df_raw.nsmallest(
    10, "salary_in_usd"
)

lowest_salaries

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
145711,2022,MI,FT,Business Intelligence Developer,15000,USD,15000,GH,100,GH,M
147922,2020,EX,FT,Staff Data Analyst,15000,USD,15000,NG,0,CA,M
150550,2021,EN,FT,Machine Learning Developer,15000,USD,15000,TH,100,TH,L
150847,2022,EN,FT,Data Analyst,15000,USD,15000,ID,0,ID,L
97343,2024,EN,FT,BI Analyst,840000,PHP,15107,PH,100,PH,M
123670,2022,EN,FT,Software Development Engineer,14400,EUR,15129,RO,50,RO,L
63638,2025,EN,FT,Data Analyst,14400,EUR,15157,FR,0,FR,M
31559,2025,EN,FT,Algorithm Developer,504000,TWD,15281,TW,0,TW,M
67087,2025,EN,FT,Algorithm Developer,504000,TWD,15281,TW,0,TW,M
68291,2025,EN,FT,Engineer,504000,TWD,15281,TW,0,TW,M


In [18]:
highest_salaries = df_raw.nlargest(
    10, "salary_in_usd"
)

highest_salaries

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
30884,2025,SE,FT,Architect,800000,USD,800000,US,0,US,M
54251,2025,SE,FT,Software Engineer,800000,USD,800000,US,0,US,M
137715,2024,MI,FT,AI Architect,800000,USD,800000,CA,100,CA,M
71362,2025,EN,FT,Data Engineer,753480,EUR,793136,AT,0,AT,M
71363,2025,EN,FT,Data Engineer,753480,EUR,793136,AT,0,AT,M
138558,2024,EN,FT,Data Analyst,774000,USD,774000,MX,0,MX,M
2228,2025,SE,FT,Machine Learning Scientist,750000,USD,750000,US,100,US,M
10303,2025,SE,FT,Data Scientist,750000,USD,750000,US,100,US,M
10305,2025,SE,FT,Data Scientist,750000,USD,750000,US,100,US,M
12954,2025,SE,FT,Data Scientist,750000,USD,750000,US,100,US,M


## Salary Outlier Assessment

In [19]:
q1 = df_raw["salary_in_usd"].quantile(0.25)
q3 = df_raw["salary_in_usd"].quantile(0.75)

iqr = q3 -q1

lower_bound = q1 - (1.5 * iqr)
upper_bound = q3 + (1.5 * iqr)

q1, q3,iqr, lower_bound, upper_bound

(np.float64(105800.0),
 np.float64(198000.0),
 np.float64(92200.0),
 np.float64(-32500.0),
 np.float64(336300.0))

In [20]:
outlier_mask = (
    (df_raw["salary_in_usd"] < lower_bound) 
    |
    (df_raw["salary_in_usd"] > upper_bound)
)

In [21]:
outlier_count = outlier_mask.sum()

outlier_percentage = (
    outlier_count / len(df_raw) * 100
)

outlier_count, round(outlier_percentage, 2)

(np.int64(3391), np.float64(2.24))

In [36]:
salary_outliers = df_raw.loc[outlier_mask]

salary_outliers_sorted = salary_outliers.sort_values(
    by= "salary_in_usd",
    ascending=False
)

salary_outliers_sorted.head(10)

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
54251,2025,SE,FT,Software Engineer,800000,USD,800000,US,0,US,M
30884,2025,SE,FT,Architect,800000,USD,800000,US,0,US,M
137715,2024,MI,FT,AI Architect,800000,USD,800000,CA,100,CA,M
71362,2025,EN,FT,Data Engineer,753480,EUR,793136,AT,0,AT,M
71363,2025,EN,FT,Data Engineer,753480,EUR,793136,AT,0,AT,M
138558,2024,EN,FT,Data Analyst,774000,USD,774000,MX,0,MX,M
38224,2025,SE,FT,Data Scientist,750000,USD,750000,US,100,US,M
61342,2025,SE,FT,Machine Learning Engineer,750000,USD,750000,US,100,US,M
61340,2025,SE,FT,Machine Learning Scientist,750000,USD,750000,US,100,US,M
10303,2025,SE,FT,Data Scientist,750000,USD,750000,US,100,US,M


### Salary Outlier Assessment

Using the IQR method, salaries above $336,300 were flagged as statistical outliers.

- Outlier records: 3,391
- Outlier percentage: 2.24%
- Decision: Retain the outliers because high salaries may represent valid senior or specialized roles.
- The median will be preferred over the mean when comparing typical salaries.

In [22]:
row_frequency = df_raw.value_counts()

row_frequency.head(10)

work_year  experience_level  employment_type  job_title                    salary  salary_currency  salary_in_usd  employee_residence  remote_ratio  company_location  company_size
2025       SE                FT               Data Scientist               160000  USD              160000         US                  100           US                M               195
                                                                           110000  USD              110000         US                  100           US                M               187
2024       SE                FT               Data Scientist               160000  USD              160000         US                  100           US                M               182
                                                                           110000  USD              110000         US                  100           US                M               168
                                              Machine Learning Researche

## Data Quality Assessment Summary

- The dataset contains 151,445 rows and 11 columns.
- No missing values were detected.
- All salaries in `salary_in_usd` are positive.
- 79,532 rows were flagged as subsequent exact matches using `duplicated()`.
- The most frequent complete row appeared 195 times.
- Exact matches were retained because the dataset has no unique respondent or job identifier. Identical attributes may represent different observations.
- Using the IQR method, 3,391 salary records (2.24%) were flagged as statistical outliers above $336,300.
- Salary outliers were retained because unusually high salaries may be valid.
- `salary_in_usd` will be used for salary comparisons.
- The median will be preferred when describing typical salaries because it is less affected by extremely high salaries.
- The dataset is heavily concentrated in 2024 and 2025, so comparisons with earlier years must be interpreted cautiously.